# Multi-Dimensional Parameter Sweeps & Trend Aggregations

This notebook covers the orchestration of large-scale sweeps across varied qubit counts and interception rates, compiling trend analysis and scientific plotting.

## 1. Running Sweep Grids
We vary qubit counts ($10, 20$) and interception rates ($0.0, 0.4$) to build a configuration matrix.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Ensure QST package is in PYTHONPATH
sys.path.insert(0, str(Path.cwd().parent / "src"))

from qst.orchestration.orchestrator import SimulationOrchestrator
from qst.orchestration.sweep_generator import ParameterSweepGenerator
from qst.models.results import SweepDimensions
from qst.analysis.aggregators.aggregator import ExperimentAggregator
from qst.analysis.trends.trends import TrendAnalysisService

qubit_counts = [10, 20]
probabilities = [0.0, 0.4]
seeds = [42]
repetitions = 2

# Generate config grid
configs = ParameterSweepGenerator.generate_configs(
    qubit_counts=qubit_counts,
    interception_probabilities=probabilities,
    seeds=seeds,
    repetitions=repetitions
)
sweep_dims = SweepDimensions(
    qubit_counts=tuple(qubit_counts),
    interception_probabilities=tuple(probabilities),
    seeds=tuple(seeds)
)

orchestrator = SimulationOrchestrator()
sweep_res = orchestrator.run_parameter_sweep(configs, sweep_dims)
print(f"Sweep completed. Total configurations executed: {sweep_res.total_experiments}")

## 2. Aggregating Metrics
We calculate the global mean metrics across the entire grid matrix.

In [ ]:
aggregator = ExperimentAggregator()
aggregated = aggregator.aggregate(sweep_res)

print(f"Total Simulations Run:   {aggregated.total_simulations}")
print(f"Global Mean QBER:        {aggregated.average_qber:.4f}")
print(f"Global Mean Key Rate:    {aggregated.average_key_rate:.4f}")
print(f"Aggregated Throughput:   {aggregated.average_throughput:.2f} qubits/sec")
print(f"Global Success Ratio:    {aggregated.success_ratio:.2f}")

## 3. Trend Analysis & Plotting
We analyze the trend lines of QBER vs. Interception rate using the TrendAnalysisService.

In [ ]:
trend_service = TrendAnalysisService()
qber_trend = trend_service.analyze_qber_vs_interception(sweep_res)

plt.figure(figsize=(8, 5))
plt.plot(qber_trend.x_values, qber_trend.y_values, marker='s', color='#2ca02c', linewidth=2, label='QBER vs Intercept')
plt.xlabel('Interception Probability')
plt.ylabel('QBER')
plt.title('Trend: QBER vs. Interception Rate')
plt.grid(True)
plt.legend()
plt.show()